# Aula 01 — Coleta e unitarização (Professor)

Notebook de condução da Aula 1, com comentários de fala, checkpoints didáticos e exportação do artefato que será consumido pela Aula 2.

## Objetivos da condução

- contextualizar a importância da comparabilidade da amostra;
- apresentar a leitura da base bruta;
- construir o valor unitário em R$/m²;
- interpretar a dispersão inicial;
- salvar a saída oficial da aula em `data/output/aula_01_amostra_unitarizada.csv`.

## Orientação ao professor

Ao longo da aula, reforce três ideias: preço absoluto não basta, unitarização cria base comparável e a análise exploratória não substitui a sanitização. Ao final, confirme em tela o caminho do arquivo exportado, porque ele será a entrada da Aula 2.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Final

import pandas as pd

from servicos.carregamento import load_raw_dataset, resolve_project_root
from servicos.unitarizacao import UNIT_PRICE_COLUMN, build_unitization_report

In [2]:
DATASET_CANDIDATES: Final[tuple[str, ...]] = (
    'amostras_residencial35.csv',
    'amostrasresidencial35.csv',
    'amostras_residencial.csv',
)


def locate_default_dataset(project_root: Path) -> Path:
    data_dir = project_root / 'data'
    for filename in DATASET_CANDIDATES:
        candidate = data_dir / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Arquivo padrão da aula não encontrado na pasta data/.')

## Etapa 1 — Abrir a base e situar a turma

Mensagem sugerida: “Antes de modelar, precisamos garantir que todos os imóveis estejam numa escala comparável”. Mostre o tamanho da base e leia as primeiras linhas com calma.

In [3]:
project_root = resolve_project_root()
dataset_path = locate_default_dataset(project_root)

df_raw = load_raw_dataset(dataset_path)
print('Projeto:', project_root)
print('Arquivo de entrada:', dataset_path)
print('Dimensão da base bruta:', df_raw.shape)
df_raw.head()

Projeto: /Users/elydocarmobarros/Desktop/ESTUDOS TEC/JUPYTER PROJECTS/treinamento_inferencia
Arquivo de entrada: /Users/elydocarmobarros/Desktop/ESTUDOS TEC/JUPYTER PROJECTS/treinamento_inferencia/data/amostras_residencial.csv
Dimensão da base bruta: (20, 7)


,id,preco,areaprivativa,vagas,idadeaparente,distanciacentrokm,fontelink
0,AP-001,750000.0,85.0,2,5.0,1.2,https://portalimoveis.com.br/anuncio/001
1,AP-002,820000.0,92.5,2,8.0,1.8,https://portalimoveis.com.br/anuncio/002
2,AP-003,690000.0,78.0,1,12.0,2.5,https://portalimoveis.com.br/anuncio/003
3,AP-004,1200000.0,115.0,3,2.0,0.8,https://portalimoveis.com.br/anuncio/004
4,AP-005,580000.0,65.0,1,20.0,4.2,https://portalimoveis.com.br/anuncio/005


## Etapa 2 — Destacar colunas relevantes

Mensagem sugerida: “Nem toda coluna entra no raciocínio ao mesmo tempo; vamos primeiro olhar preço, área e variáveis de caracterização”.

In [4]:
preview_columns = [
    column for column in ('id', 'preco', 'areaprivativa', 'vagas', 'idadeaparente', 'distanciacentrokm')
    if column in df_raw.columns
]

df_raw[preview_columns].head(10)

,id,preco,areaprivativa,vagas,idadeaparente,distanciacentrokm
0,AP-001,750000.0,85.0,2,5.0,1.2
1,AP-002,820000.0,92.5,2,8.0,1.8
2,AP-003,690000.0,78.0,1,12.0,2.5
3,AP-004,1200000.0,115.0,3,2.0,0.8
4,AP-005,580000.0,65.0,1,20.0,4.2
5,AP-006,950000.0,105.0,2,4.0,1.5
6,AP-007,710000.0,80.0,1,10.0,2.1
7,AP-008,640000.0,72.0,1,15.0,3.0
8,AP-009,2400000.0,120.0,3,1.0,0.5
9,AP-010,880000.0,98.0,2,6.0,1.9


## Etapa 3 — Unitarizar a amostra

Mensagem sugerida: “Agora o preço total passa a ser lido em termos de preço por área, o que melhora a comparabilidade entre imóveis de tamanhos diferentes”.

In [5]:
report = build_unitization_report(df_raw)
df_unitized = report['dataframe_unitarizado']

print('Total de amostras:', report['total_amostras'])
print(f"Média do valor unitário: {report['media_valor_unitario']:.2f}")
print(f"Mediana do valor unitário: {report['mediana_valor_unitario']:.2f}")
print(f"Desvio-padrão do valor unitário: {report['desvio_padrao_valor_unitario']:.2f}")
print(f"Coeficiente de variação do valor unitário bruto: {report['coeficiente_variacao_percentual']}%")
df_unitized.head()

Total de amostras: 20
Média do valor unitário: 9334.19
Mediana do valor unitário: 8962.04
Desvio-padrão do valor unitário: 2771.92
Coeficiente de variação do valor unitário bruto: 29.7%


,id,preco,areaprivativa,vagas,idadeaparente,distanciacentrokm,fontelink,valor_unitario
0,AP-001,750000.0,85.0,2,5.0,1.2,https://portalimoveis.com.br/anuncio/001,8823.529412
1,AP-002,820000.0,92.5,2,8.0,1.8,https://portalimoveis.com.br/anuncio/002,8864.864865
2,AP-003,690000.0,78.0,1,12.0,2.5,https://portalimoveis.com.br/anuncio/003,8846.153846
3,AP-004,1200000.0,115.0,3,2.0,0.8,https://portalimoveis.com.br/anuncio/004,10434.782609
4,AP-005,580000.0,65.0,1,20.0,4.2,https://portalimoveis.com.br/anuncio/005,8923.076923


## Etapa 4 — Leitura didática dos resultados

Use esta etapa para verbalizar se a dispersão inicial parece alta ou moderada. Não antecipe exclusões ainda; apenas diga que a próxima aula fará a sanitização com critério estatístico.

In [6]:
summary_table = pd.DataFrame(
    {
        'indicador': [
            'amostras',
            'media_unitaria',
            'mediana_unitaria',
            'desvio_padrao_unitario',
            'cv_percentual',
        ],
        'valor': [
            report['total_amostras'],
            report['media_valor_unitario'],
            report['mediana_valor_unitario'],
            report['desvio_padrao_valor_unitario'],
            report['coeficiente_variacao_percentual'],
        ],
    }
)
summary_table

,indicador,valor
0,amostras,20.000000
1,media_unitaria,9334.191143
2,mediana_unitaria,8962.038304
3,desvio_padrao_unitario,2771.918277
4,cv_percentual,29.700000


## Etapa 5 — Exportar a saída oficial da Aula 1

Feche a aula mostrando que o arquivo exportado será a entrada da Aula 2. Esse é o ponto de conexão entre os notebooks.

In [7]:
output_dir = project_root / 'data' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'aula_01_amostra_unitarizada.csv'
df_unitized.to_csv(output_path, index=False)
print('Arquivo exportado:', output_path)

Arquivo exportado: /Users/elydocarmobarros/Desktop/ESTUDOS TEC/JUPYTER PROJECTS/treinamento_inferencia/data/output/aula_01_amostra_unitarizada.csv


## Fechamento

Encerramento sugerido: “A Aula 1 termina com a base unitarizada e pronta para a sanitização. Na Aula 2, o arquivo gerado aqui será lido e submetido ao critério de Chauvenet”.